<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 5 · Árboles y *boosting* desde cero

**Unidad 5**

Un árbol de decisión es una sucesión de preguntas del tipo «¿la hora es anterior a las 7:30?», y aprenderlo consiste en elegir, una y otra vez, la pregunta que mejor separa los datos. En este laboratorio vas a construir esa maquinaria pieza por pieza: las medidas de **impureza**, la búsqueda de la **mejor división** de un atributo en tiempo $O(N \log N)$, un **árbol de regresión** recursivo completo, y luego los dos ensambles que lo vuelven competitivo: el ***bagging***, con su estimación del error **fuera de la bolsa** (OOB), y el ***gradient boosting***. Terminamos con las dos fórmulas que hacen distinto a **XGBoost**: el peso óptimo de una hoja y la ganancia de una división calculados con el gradiente y la hessiana de la pérdida.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** El árbol de las preguntas 1 a 3 y las fórmulas de la pregunta 6 se programan solo con numpy: nada de `sklearn` ni `xgboost`, que ahí se usan únicamente para comparar. En las preguntas 4 y 5 (los ensambles) el aprendiz base es `DecisionTreeRegressor` de scikit-learn, porque el tuyo, escrito en Python, sería demasiado lento para entrenar cientos de árboles. Las operaciones sobre los datos deben ser vectorizadas: no recorras los ejemplos con ciclos. Los ciclos sobre atributos, sobre nodos (la recursión) y sobre árboles del ensamble sí están permitidos.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "rediseno-2027"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u05-lab05-arboles-boosting"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/{RAMA}/unidades/05-arboles-ensambles/laboratorios/tests/u05-lab05-arboles-boosting/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

import rdp

rdp.estilo.usar()

## Los datos y el problema

Volvemos a las **bicicletas compartidas de Washington** de la Unidad 4: 17,379 horas de 2011 y 2012 del sistema Capital Bikeshare, con el número de rentas en cada hora, el calendario y el clima (H. Fanaee-T y J. Gama, *Progress in Artificial Intelligence*, 2014; repositorio UCI, licencia CC BY 4.0).

**El problema** es el mismo: predecir `total`, las rentas de una hora, a partir de la hora del día, el calendario y el clima. Usamos la misma partición que en la Unidad 4: entrenamos con los días 1 a 19 de cada mes (10,886 horas) y probamos con los días 20 en adelante (6,493 horas), para que el conjunto de prueba sean días completos que el modelo nunca vio.

En la Unidad 4 la regresión lineal tuvo que recibir **a mano** la interacción entre la hora y el tipo de día para bajar su raíz del error cuadrático medio (RECM) de prueba de 165.0 a 86.4, y la regresión de Poisson llegó a 72.2. Un árbol no necesita esa ayuda: pregunta primero por la hora y después, dentro de cada rama, por el tipo de día, y así descubre la interacción solo. Tampoco hay que estandarizar ni codificar nada: a un árbol solo le importa el **orden** de los valores de cada atributo.

In [ ]:
b = rdp.datos.cargar("bicis_washington", parse_dates=["fecha"])
b["anio"] = b.fecha.dt.year - 2011          # 0 = 2011, 1 = 2012
b["clima"] = b.clima.clip(upper=3)          # la categoría 4 tiene 3 horas: se une a la 3
entrena = b[b.fecha.dt.day <= 19]
prueba = b[b.fecha.dt.day >= 20]

atributos = ["hora", "laboral", "dia_semana", "anio", "mes", "clima", "temp", "sensacion", "humedad", "viento"]
X_ent, y_ent = entrena[atributos].to_numpy(dtype=float), entrena["total"].to_numpy(dtype=float)
X_pru, y_pru = prueba[atributos].to_numpy(dtype=float), prueba["total"].to_numpy(dtype=float)


def recm(y, y_hat):
    """Raíz del error cuadrático medio."""
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(y_hat)) ** 2)))


print(f"entrenamiento: {X_ent.shape} · prueba: {X_pru.shape}")
print(f"RECM de predecir siempre la media: {recm(y_pru, y_ent.mean()):.1f}")

## 1. Impureza: Gini y entropía

Empecemos por la clasificación, donde se inventaron estas medidas. En un nodo con $n_k$ ejemplos de la clase $k$ ($k = 1, \dots, K$) y proporciones $p_k = n_k / \sum_j n_j$, las dos impurezas más usadas son

$$G = 1 - \sum_{k=1}^{K} p_k^2 \qquad\text{y}\qquad H = -\sum_{k=1}^{K} p_k \log_2 p_k ,$$

con la convención $0 \log 0 = 0$ (el límite de $p \log p$ cuando $p \to 0$). Las dos valen 0 en un nodo puro y son máximas cuando las clases están parejas. El índice de Gini tiene una lectura directa: es la probabilidad de equivocarse si a un ejemplo del nodo se le asigna una clase **al azar** con las probabilidades $p_k$, porque $\sum_k p_k(1 - p_k) = 1 - \sum_k p_k^2$.

Un árbol evalúa miles de divisiones candidatas, así que conviene calcular la impureza de muchos nodos a la vez.

**Pregunta 1.** Escribe `gini(conteos)` y `entropia(conteos)`. `conteos` es un arreglo de enteros no negativos de cualquier forma cuyo **último eje** son las $K$ clases (por ejemplo, de $M \times K$: un renglón por nodo). Cada función devuelve un arreglo con la forma de los demás ejes (de longitud $M$ en el ejemplo). La entropía va en **bits** (logaritmo base 2). Un nodo vacío (todos los conteos en cero) tiene impureza 0. No debe aparecer ningún aviso de división entre cero ni de $\log 0$: `np.where` no basta, porque calcula las dos ramas antes de elegir; usa el argumento `where=` de las funciones de numpy (por ejemplo `np.log2(p, out=np.zeros_like(p), where=p > 0)`) o `np.divide`.

In [ ]:
def gini(conteos):
    """Índice de Gini de cada nodo; el último eje son las clases."""
    ...


def entropia(conteos):
    """Entropía (en bits) de cada nodo; el último eje son las clases."""
    ...


ejemplo = np.array([[50, 50], [90, 10], [100, 0], [0, 0]])
print("Gini:    ", gini(ejemplo).round(4))
print("entropía:", entropia(ejemplo).round(4))

In [ ]:
grader.check("p1")

La **ganancia** de una división es la impureza del padre menos el promedio de las impurezas de las hijas, ponderado por su tamaño. ¿Por qué no usar simplemente la fracción de errores del nodo, $1 - \max_k p_k$, que es lo que al final nos importa? El ejemplo clásico: un nodo con 400 ejemplos de cada clase y dos divisiones posibles.

- **A**: hijas con conteos (300, 100) y (100, 300).
- **B**: hijas con conteos (200, 400) y (200, 0).

Las dos se equivocan en 200 de 800 ejemplos, así que el error de clasificación no las distingue. Pero B produce un nodo **puro**, que ya no hay que seguir dividiendo, y la otra hija todavía se puede separar después. Gini y entropía, que son estrictamente cóncavas, sí prefieren B:

In [ ]:
padre = np.array([400, 400])
divisiones = {"A": np.array([[300, 100], [100, 300]]), "B": np.array([[200, 400], [200, 0]])}
for nombre, hijas in divisiones.items():
    pesos = hijas.sum(axis=1) / padre.sum()
    error = lambda c: 1 - c.max(axis=-1) / c.sum(axis=-1)
    print(f"{nombre}: ganancia en error = {error(padre) - pesos @ error(hijas):.4f} · "
          f"en Gini = {gini(padre) - pesos @ gini(hijas):.4f} · en entropía = {entropia(padre) - pesos @ entropia(hijas):.4f}")

## 2. La mejor división de un atributo, ordenando una sola vez

En **regresión** la impureza de un nodo es la suma de cuadrados alrededor de su media, $\mathrm{SC} = \sum_i (y_i - \bar y)^2$, y la hoja predice $\bar y$. Para un atributo $x$, una división «$x \le u$» manda a la izquierda $I = \{i : x_i \le u\}$ y a la derecha el resto, y su **reducción** es

$$\Delta(u) = \mathrm{SC}_{\text{padre}} - \mathrm{SC}_I - \mathrm{SC}_D .$$

Probar cada umbral recalculando las dos sumas cuesta $O(N)$ por umbral y $O(N^2)$ en total. El truco es usar que $\mathrm{SC} = \sum_i y_i^2 - S^2/n$, donde $S$ es la suma de las $y$ del nodo y $n$ su tamaño. Los $\sum y_i^2$ del padre y de las hijas se cancelan en $\Delta$, y queda

$$\Delta = \frac{S_I^2}{n_I} + \frac{S_D^2}{n_D} - \frac{S^2}{n}.$$

Si **ordenamos** los datos por $x$, las particiones posibles son «los primeros $k$ a la izquierda» para $k = 1, \dots, N - 1$, y $S_I$ para todas ellas es la **suma acumulada** de las $y$ ordenadas: todo cuesta $O(N \log N)$, por el ordenamiento.

Dos detalles. Solo se puede cortar entre dos valores **distintos** de $x$ (si $x_k = x_{k+1}$, ningún umbral los separa), y el umbral es el **punto medio** $u = (x_k + x_{k+1})/2$, como en scikit-learn. Además, `min_hoja` exige que cada hija tenga al menos ese número de ejemplos. Para no perder precisión al restar números grandes, conviene **centrar** las $y$ (restarles su media) antes de acumular: $\Delta$ no cambia.

**Pregunta 2.** Escribe `mejor_division(x, y, min_hoja=1)`, que devuelve la tupla `(umbral, reduccion)` de la división de mayor $\Delta$ (como `float`). Si hay empate, gana el umbral más pequeño. Si no hay ninguna división válida (todos los $x$ iguales, o muy pocos datos para `min_hoja`), devuelve `(np.nan, 0.0)`. Ordena una sola vez (`np.argsort`) y usa `np.cumsum`: nada de ciclos sobre los umbrales.

In [ ]:
def mejor_division(x, y, min_hoja=1):
    """Mejor umbral u para dividir en x <= u (regresión, suma de cuadrados). Devuelve (umbral, reduccion)."""
    ...


print("hora:", mejor_division(X_ent[:, 0], y_ent))
print("temp:", mejor_division(X_ent[:, 6], y_ent))

In [ ]:
grader.check("p2")

## 3. Un árbol de regresión completo

El algoritmo CART crece el árbol de manera **voraz** y recursiva: en cada nodo busca, entre todos los atributos, la división de mayor reducción, parte los datos y repite en cada hija. Encontrar el árbol óptimo es NP-completo (Hyafil y Rivest, 1976), así que nadie lo intenta; el voraz funciona bien en la práctica.

Representaremos el árbol con diccionarios anidados:

- una **hoja** es `{"valor": v}`, donde `v` es el promedio de las $y$ que llegaron a ella;
- un **nodo interno** es `{"atributo": j, "umbral": u, "izq": ..., "der": ...}`; los ejemplos con `X[:, j] <= u` van a `"izq"` y los demás a `"der"`.

**Pregunta 3.** Escribe `ajustar_arbol(X, y, max_depth=3, min_samples_leaf=1)`, que devuelve el árbol, y `predecir_arbol(arbol, X)`, que devuelve un arreglo con una predicción por renglón de `X`.

Un nodo se vuelve **hoja** si ya se alcanzó la profundidad máxima (`max_depth=0` da una sola hoja), si tiene menos de `2 * min_samples_leaf` ejemplos, o si ningún atributo tiene una división válida con reducción **positiva**. Si no, se divide con el atributo de mayor reducción (en un empate, el de menor índice) usando tu `mejor_division` con `min_hoja=min_samples_leaf`. En `predecir_arbol` no recorras los renglones uno por uno: manda **todos** los que caen en cada rama juntos, con una máscara booleana, y recursa.

In [ ]:
def ajustar_arbol(X, y, max_depth=3, min_samples_leaf=1):
    """Árbol de regresión CART (suma de cuadrados) como diccionarios anidados."""
    ...


def predecir_arbol(arbol, X):
    """Predicciones del árbol para cada renglón de X."""
    ...


def mostrar(arbol, nombres, sangria=""):
    """Imprime el árbol como reglas anidadas."""
    if "atributo" not in arbol:
        print(f"{sangria}→ {arbol['valor']:.1f} rentas")
        return
    nombre, u = nombres[arbol["atributo"]], arbol["umbral"]
    print(f"{sangria}si {nombre} ≤ {u:g}:")
    mostrar(arbol["izq"], nombres, sangria + "    ")
    print(f"{sangria}si {nombre} > {u:g}:")
    mostrar(arbol["der"], nombres, sangria + "    ")


arbol_3 = ajustar_arbol(X_ent, y_ent, max_depth=3)
mostrar(arbol_3, atributos)

In [ ]:
grader.check("p3")

Lee el árbol de profundidad 3 de arriba. La primera pregunta separa la madrugada del resto del día, y a las 6 de la mañana el árbol pregunta por `laboral`: en un día laboral espera unas 103 rentas y en un fin de semana o feriado, 20. Es la interacción hora × día laboral que en la Unidad 4 construimos a mano, descubierta sola. Con más profundidad aparece también en los picos de la mañana y de la tarde.

¿Qué tan profundo conviene crecerlo? La celda siguiente entrena árboles de profundidad 1 a 16 (con `min_samples_leaf=5`) y mide la RECM en entrenamiento y en prueba. Tarda unos segundos.

In [ ]:
profundidades = list(range(1, 17))
error_ent, error_pru = [], []
inicio = time.perf_counter()
for d in profundidades:
    a = ajustar_arbol(X_ent, y_ent, max_depth=d, min_samples_leaf=5)
    error_ent.append(recm(y_ent, predecir_arbol(a, X_ent)))
    error_pru.append(recm(y_pru, predecir_arbol(a, X_pru)))
print(f"tiempo: {time.perf_counter() - inicio:.1f} s")
mejor_d = profundidades[int(np.argmin(error_pru))]
print(f"mejor profundidad en prueba: {mejor_d} · RECM de prueba: {min(error_pru):.1f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(profundidades, error_ent, color=rdp.estilo.AZUL, marker="o", ms=4)
ax.plot(profundidades, error_pru, color=rdp.estilo.ROJO, marker="o", ms=4)
ax.axhline(72.2, color=rdp.estilo.GRIS, ls="--", lw=1)
ax.text(1.2, 75, "Poisson de la Unidad 4 (72.2)", color=rdp.estilo.GRIS, fontsize=10)
ax.set_xlim(0.5, 19)
ax.set_xticks(range(2, 17, 2))
rdp.estilo.etiquetar(ax, 16.4, {"entrenamiento": error_ent[-1], "prueba": error_pru[-1]})
ax.set_title("Un árbol de regresión: error contra profundidad")
ax.set_xlabel("profundidad máxima")
ax.set_ylabel("RECM (rentas por hora)");

La forma ya la conoces de la Unidad 3: el error de entrenamiento baja sin parar y el de prueba se estanca. El mejor árbol solo se queda en unos 78 y no alcanza a la regresión de Poisson. Los ensambles de las siguientes preguntas sí la superan.

## 4. *Bagging* y el error fuera de la bolsa

Un árbol profundo tiene poco sesgo pero mucha varianza: si cambian un poco los datos, cambia mucho. El ***bagging*** (*bootstrap aggregating*, Breiman, 1996) entrena $B$ árboles, cada uno con una **remuestra *bootstrap***: $N$ índices elegidos al azar **con reemplazo** de entre los $N$ ejemplos de entrenamiento. La predicción es el promedio de los $B$ árboles.

En cada remuestra, la probabilidad de que un ejemplo dado no salga nunca es $(1 - 1/N)^N \approx e^{-1} \approx 0.368$. Para cada ejemplo $i$ hay entonces unos $0.37\,B$ árboles que **no** lo vieron, y el promedio de sus predicciones es una predicción «honesta» de $y_i$: es la **predicción fuera de la bolsa** (*out of bag*, OOB). Con ella se estima el error de prueba sin apartar datos y casi sin costo.

Para que tu resultado sea reproducible, sigue exactamente esta receta:

1. `rng = np.random.default_rng(semilla)` y `muestras = rng.integers(0, N, size=(B, N))`, una remuestra por renglón;
2. el árbol $b$ es `DecisionTreeRegressor(random_state=0, **params).fit(X[muestras[b]], y[muestras[b]])`.

**Pregunta 4.** Escribe tres funciones:

- `ajustar_bagging(X, y, B=50, semilla=0, **params)` devuelve la tupla `(arboles, muestras)`: la lista de los $B$ árboles y el arreglo de índices de $B \times N$, siguiendo la receta de arriba (`params` son los parámetros del árbol, como `max_depth`);
- `predecir_bagging(arboles, X)` devuelve el promedio de las predicciones de los árboles;
- `prediccion_oob(arboles, muestras, X)` recibe la `X` de **entrenamiento** y devuelve, para cada ejemplo, el promedio de las predicciones de los árboles en cuya remuestra **no** aparece; `np.nan` si apareció en todas.

En `prediccion_oob` puedes recorrer los árboles, pero no los ejemplos. Pista: arma una matriz booleana `en_bolsa` de $B \times N$. Cuidado con un árbol cuya remuestra contiene a **todos** los ejemplos: scikit-learn no acepta predecir sobre cero renglones.

In [ ]:
def ajustar_bagging(X, y, B=50, semilla=0, **params):
    """Bagging de árboles de regresión. Devuelve (arboles, muestras)."""
    ...


def predecir_bagging(arboles, X):
    """Promedio de las predicciones de los árboles."""
    ...


def prediccion_oob(arboles, muestras, X):
    """Predicción fuera de la bolsa de cada ejemplo de entrenamiento (nan si no hay árboles que no lo vieran)."""
    ...


inicio = time.perf_counter()
arboles_bag, muestras_bag = ajustar_bagging(X_ent, y_ent, B=100, semilla=2011, min_samples_leaf=1)
print(f"tiempo: {time.perf_counter() - inicio:.1f} s")
pred_oob = prediccion_oob(arboles_bag, muestras_bag, X_ent)
print(f"RECM fuera de la bolsa: {recm(y_ent, pred_oob):.1f} · RECM de prueba: {recm(y_pru, predecir_bagging(arboles_bag, X_pru)):.1f}")

In [ ]:
grader.check("p4")

¿Cuántos árboles hacen falta? La celda siguiente calcula la RECM de prueba y la OOB usando solo los primeros $b$ árboles, para $b = 1, \dots, 100$.

In [ ]:
P_pru = np.array([a.predict(X_pru) for a in arboles_bag])
pru_por_b = [recm(y_pru, P_pru[:k].mean(axis=0)) for k in range(1, 101)]
oob_por_b = []
for k in range(1, 101):
    o = prediccion_oob(arboles_bag[:k], muestras_bag[:k], X_ent)
    oob_por_b.append(recm(y_ent[~np.isnan(o)], o[~np.isnan(o)]))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, 101), pru_por_b, color=rdp.estilo.ROJO)
ax.plot(range(1, 101), oob_por_b, color=rdp.estilo.AZUL)
ax.axhline(72.2, color=rdp.estilo.GRIS, ls="--", lw=1)
ax.text(60, 74, "Poisson de la Unidad 4 (72.2)", color=rdp.estilo.GRIS, fontsize=10)
ax.set_xlim(0, 118)
rdp.estilo.etiquetar(ax, 102, {"prueba": pru_por_b[-1], "fuera de la bolsa": oob_por_b[-1]})
ax.set_title("Bagging de árboles profundos: error contra número de árboles")
ax.set_xlabel("número de árboles")
ax.set_ylabel("RECM (rentas por hora)");

Promediar árboles baja la RECM de prueba de unos 82 con un solo árbol sin podar (entrenado con una remuestra) a unos 66, por debajo de Poisson, y la mejora se aplana después de unas decenas de árboles. Pero mira la distancia entre las dos curvas: la OOB promete bastante menos error del que vemos en prueba. La reflexión 2 te pide explicar por qué.

## 5. *Gradient boosting* con pérdida cuadrática

El *bagging* promedia árboles independientes; el ***boosting*** los construye **en secuencia**, cada uno corrigiendo lo que les falta a los anteriores. Con pérdida cuadrática $\frac12 (y - F)^2$ la receta de Friedman (2001) es:

1. empieza con la constante $F_0(\mathbf{x}) = \bar y$;
2. para $m = 1, \dots, M$: calcula los **residuos** $r_i = y_i - F_{m-1}(\mathbf{x}_i)$, ajusta un árbol pequeño $h_m$ a los pares $(\mathbf{x}_i, r_i)$ y actualiza $F_m = F_{m-1} + \nu\, h_m$.

Los residuos son menos el gradiente de la pérdida respecto a $F$: cada árbol da un paso de **descenso por gradiente en el espacio de funciones**. Con otras pérdidas, $r_i$ se reemplaza por $-\partial L / \partial F$ evaluado en $F_{m-1}(\mathbf{x}_i)$, los **pseudo-residuos**. La **tasa de aprendizaje** $\nu \in (0, 1]$ (*shrinkage*) hace que cada árbol corrija solo una parte del error: se necesitan más árboles, pero el modelo generaliza mejor.

Usa como árbol `DecisionTreeRegressor(max_depth=max_depth, random_state=0)`.

**Pregunta 5.** Escribe `ajustar_gb(X, y, n_arboles=100, tasa=0.1, max_depth=3)`, que devuelve la tupla `(f0, arboles)` con la constante inicial (un `float`) y la lista de árboles, y `predecir_gb(f0, arboles, X, tasa=0.1)`, que devuelve $F_M(\mathbf{x})$ para cada renglón de `X`.

In [ ]:
def ajustar_gb(X, y, n_arboles=100, tasa=0.1, max_depth=3):
    """Gradient boosting con pérdida cuadrática. Devuelve (f0, arboles)."""
    ...


def predecir_gb(f0, arboles, X, tasa=0.1):
    """Predicción del modelo aditivo f0 + tasa · Σ h_m(x)."""
    ...


inicio = time.perf_counter()
f0_gb, arboles_gb = ajustar_gb(X_ent, y_ent, n_arboles=300, tasa=0.1, max_depth=4)
print(f"tiempo: {time.perf_counter() - inicio:.1f} s")
print(f"RECM de prueba (300 árboles, tasa 0.1, profundidad 4): {recm(y_pru, predecir_gb(f0_gb, arboles_gb, X_pru, tasa=0.1)):.1f}")

In [ ]:
grader.check("p5")

El compromiso entre la tasa y el número de árboles se ve mejor en una gráfica. Con la tasa 1 (sin *shrinkage*) el error de prueba baja muy rápido y luego **sube**: el modelo empieza a ajustar el ruido. Con tasas más chicas baja más despacio, pero llega más abajo (cerca de 63.5) y en 300 árboles todavía no empeora. En la práctica se fija una tasa pequeña y se elige el número de árboles con un conjunto de validación, deteniéndose cuando el error deja de bajar: la **parada temprana**. (Aquí usamos el de prueba solo para ilustrar; para elegir, nunca.)

In [ ]:
curvas = {}
for nu in [1.0, 0.3, 0.1]:
    f0, arbs = ajustar_gb(X_ent, y_ent, n_arboles=300, tasa=nu, max_depth=4)
    F = np.full(len(X_pru), f0)
    errores = []
    for h in arbs:
        F = F + nu * h.predict(X_pru)
        errores.append(recm(y_pru, F))
    curvas[f"tasa {nu:g}"] = errores
    print(f"tasa {nu:g}: mínimo {min(errores):.1f} con {int(np.argmin(errores)) + 1} árboles · con 300: {errores[-1]:.1f}")

fig, ax = plt.subplots(figsize=(9, 5))
for (nombre, errores), color in zip(curvas.items(), [rdp.estilo.ROJO, rdp.estilo.DORADO_OSC, rdp.estilo.AZUL]):
    ax.plot(range(1, 301), errores, color=color)
ax.set_ylim(55, 110)
ax.set_xlim(0, 350)
rdp.estilo.etiquetar(ax, 305, {nombre: errores[-1] for nombre, errores in curvas.items()})
ax.set_title("Gradient boosting: error de prueba contra número de árboles")
ax.set_xlabel("número de árboles")
ax.set_ylabel("RECM de prueba (rentas por hora)");

## 6. La hoja y la ganancia de XGBoost

XGBoost (Chen y Guestrin, 2016) cambia dos cosas. Primero, en vez de ajustar el árbol a los pseudo-residuos con mínimos cuadrados, aproxima la pérdida alrededor de la predicción actual con su **desarrollo de Taylor de segundo orden**. Con $g_i = \partial L/\partial F$ y $h_i = \partial^2 L/\partial F^2$ evaluadas en $F_{m-1}(\mathbf{x}_i)$, sumar $w$ a la predicción de los ejemplos de una hoja cambia la pérdida en

$$\sum_{i \in \text{hoja}} \Bigl(g_i w + \tfrac12 h_i w^2\Bigr) + \tfrac{\lambda}{2} w^2 = G w + \tfrac12 (H + \lambda) w^2 ,$$

donde $G$ y $H$ son las sumas de los $g_i$ y los $h_i$ de la hoja. Segundo, **regulariza** el árbol: $\lambda$ penaliza pesos grandes (como *ridge*) y $\gamma$ cobra por cada hoja. Minimizando la parábola en $w$:

$$w^* = -\frac{G}{H + \lambda}, \qquad \text{y la pérdida baja en}\quad \frac12\,\frac{G^2}{H + \lambda}.$$

Al dividir una hoja en dos hijas, $I$ y $D$, se pierde la hoja del padre y se ganan dos, y se paga una hoja más:

$$\text{ganancia} = \frac12\left[\frac{G_I^2}{H_I + \lambda} + \frac{G_D^2}{H_D + \lambda} - \frac{(G_I + G_D)^2}{H_I + H_D + \lambda}\right] - \gamma .$$

Con pérdida cuadrática, $g_i = F_{m-1}(\mathbf{x}_i) - y_i$ y $h_i = 1$: el peso $w^*$ es la suma de los residuos dividida entre $n + \lambda$, un promedio «encogido» hacia cero, y la ganancia con $\lambda = \gamma = 0$ es exactamente la mitad de la reducción $\Delta$ de la pregunta 2. Con pérdida logística, $h_i = p_i(1 - p_i)$ y las hojas dan un paso de Newton.

**Pregunta 6.** Escribe `peso_hoja(G, H, lam=1.0)` y `ganancia_xgb(G_izq, H_izq, G_der, H_der, lam=1.0, gamma=0.0)`, vectorizadas (sus argumentos pueden ser arreglos). Después escribe `mejor_division_xgb(x, g, h, lam=1.0, gamma=0.0)`, la versión de XGBoost de tu `mejor_division`: ordena una vez, acumula $G_I$ y $H_I$, y devuelve `(umbral, ganancia)` de la división de mayor ganancia entre valores distintos de $x$ (umbral en el punto medio; en un empate, el más pequeño), aunque esa ganancia sea negativa; `(np.nan, 0.0)` si todos los $x$ son iguales.

In [ ]:
def peso_hoja(G, H, lam=1.0):
    """Peso óptimo de una hoja de XGBoost."""
    ...


def ganancia_xgb(G_izq, H_izq, G_der, H_der, lam=1.0, gamma=0.0):
    """Ganancia de XGBoost al dividir un nodo en dos hijas."""
    ...


def mejor_division_xgb(x, g, h, lam=1.0, gamma=0.0):
    """Mejor umbral de un atributo según la ganancia de XGBoost. Devuelve (umbral, ganancia)."""
    ...


# primer árbol de XGBoost en las bicicletas, con pérdida cuadrática y F_0 = media
g_bicis = y_ent.mean() - y_ent
h_bicis = np.ones_like(y_ent)
for lam in [0.0, 1.0, 1000.0]:
    u, gan = mejor_division_xgb(X_ent[:, 0], g_bicis, h_bicis, lam=lam)
    izq = X_ent[:, 0] <= u
    print(f"λ = {lam:>6g}: hora ≤ {u} · ganancia {gan:,.0f} · pesos de las hojas "
          f"{peso_hoja(g_bicis[izq].sum(), izq.sum(), lam):+.1f} y {peso_hoja(g_bicis[~izq].sum(), (~izq).sum(), lam):+.1f}")

In [ ]:
grader.check("p6")

Con $\lambda = 0$ los pesos son los promedios de los residuos en cada hoja, igual que en el *gradient boosting* de la pregunta 5. Con $\lambda = 1000$ se encogen hacia cero y la ganancia baja, pero en este caso la división elegida no cambia porque las dos hojas tienen miles de horas. Donde $\lambda$ sí cambia las cosas es en las hojas con pocos ejemplos, que son justamente las que más sobreajustan.

## 7. Resumen

La tabla junta los modelos de este laboratorio con los de la Unidad 4.

In [ ]:
mejor_arbol = ajustar_arbol(X_ent, y_ent, max_depth=mejor_d, min_samples_leaf=5)
resumen = {
    "lineal con interacción (Unidad 4)": 86.4,
    "Poisson (Unidad 4)": 72.2,
    f"tu árbol (profundidad {mejor_d})": recm(y_pru, predecir_arbol(mejor_arbol, X_pru)),
    "bagging, 100 árboles": recm(y_pru, predecir_bagging(arboles_bag, X_pru)),
    "gradient boosting, 300 árboles": recm(y_pru, predecir_gb(f0_gb, arboles_gb, X_pru, tasa=0.1)),
}
for nombre, valor in resumen.items():
    print(f"{nombre:>36}: {valor:6.1f}")

## 8. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una. Si necesitas hacer cuentas, agrega celdas de código.

<!-- BEGIN QUESTION -->

**Reflexión 1.** ¿Por qué un solo árbol no generaliza tan bien como el *bagging* de árboles, si cada árbol del ensamble es, por sí solo, **peor** que tu mejor árbol (se entrenó con una remuestra y sin podar)? Para responder, entrena 30 árboles profundos con 30 remuestras distintas (puedes usar los de `arboles_bag`), toma tres horas de prueba y mira cuánto varían sus predicciones de un árbol a otro. Usa la fórmula de la varianza del promedio de $B$ variables con varianza $\sigma^2$ y correlación $\rho$, $\rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$, para explicar qué parte del error reduce el promedio, qué parte no, y por qué los **bosques aleatorios** eligen un subconjunto aleatorio de atributos en cada división. ¿Por qué el *boosting*, en cambio, usa árboles **pequeños**?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** En la sección 4, la RECM fuera de la bolsa salió bastante **menor** que la de prueba, aunque la teoría dice que la OOB es un estimador casi insesgado del error de prueba. ¿Qué supuesto de esa teoría no se cumple con estos datos? Piensa en qué ejemplos de entrenamiento «se parecen» a una hora OOB y en cómo se partieron los datos en entrenamiento y prueba (ya lo discutimos en la Unidad 4). Propón una manera de hacer remuestras para que la estimación OOB sea más honesta aquí y, si quieres, pruébala.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()